이 노트북은 <[케라스 창시자에게 배우는 딥러닝 (개정 3판)](https://tensorflow.blog/kerasdl3/)>(길벗, 2026)의 예제 코드를 담고 있습니다.

### 파이토치 소개

**준비:** 원본 노트북의 텐서플로 절에서 만든 데이터를 이 노트북에서도 쓰므로 여기서 다시 만듭니다.

In [3]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [4]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [5]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

#### 파이토치 첫걸음

##### 파이토치 텐서와 파라미터

###### 상수 텐서

In [6]:
import torch
torch.ones(size=(2, 1))

tensor([[1.],
        [1.]])

In [7]:
torch.zeros(size=(2, 1))

tensor([[0.],
        [0.]])

In [8]:
torch.tensor([1, 2, 3], dtype=torch.float32)

tensor([1., 2., 3.])

###### 랜덤 텐서

In [9]:
torch.normal(
mean=torch.zeros(size=(3, 1)),
std=torch.ones(size=(3, 1)))

tensor([[0.2439],
        [1.0315],
        [1.0914]])

In [10]:
torch.rand(3, 1)

tensor([[0.9469],
        [0.0466],
        [0.6997]])

###### 텐서 할당과 Parameter 클래스

In [11]:
x = torch.zeros(size=(2, 1))
x[0, 0] = 1.
x

tensor([[1.],
        [0.]])

In [12]:
x = torch.zeros(size=(2, 1))
p = torch.nn.parameter.Parameter(data=x)

##### 텐서 연산: 파이토치로 수학 계산하기

In [13]:
a = torch.ones((2, 2))
b = torch.square(a)
c = torch.sqrt(a)
d = b + c
e = torch.matmul(a, b)
f = torch.cat((a, b), dim=0)

In [14]:
def dense(inputs, W, b):
    return torch.nn.relu(torch.matmul(inputs, W) + b)

##### 파이토치로 그레이디언트 계산하기

In [15]:
input_var = torch.tensor(3.0, requires_grad=True)
result = torch.square(input_var)
result.backward()
gradient = input_var.grad
gradient

tensor(6.)

In [16]:
result = torch.square(input_var)
result.backward()
input_var.grad

tensor(12.)

In [17]:
input_var.grad = None

#### 엔드-투-엔드 예제: 순수 파이토치로 구현하는 선형 분류기

In [18]:
input_dim = 2
output_dim = 1

W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

In [19]:
def model(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [20]:
def mean_squared_error(targets, predictions):
    per_sample_losses = torch.square(targets - predictions)
    return torch.mean(per_sample_losses)

In [21]:
learning_rate = 0.1

def training_step(inputs, targets, W, b):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    grad_loss_wrt_W, grad_loss_wrt_b = W.grad, b.grad
    with torch.no_grad():
        W -= grad_loss_wrt_W * learning_rate
        b -= grad_loss_wrt_b * learning_rate
    W.grad = None
    b.grad = None
    return loss

##### Module 클래스로 상태과 계산 패키징하기

In [22]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.zeros(output_dim))

    def forward(self, inputs):
        return torch.matmul(inputs, self.W) + self.b

In [23]:
model = LinearModel()

In [24]:
torch_inputs = torch.tensor(inputs)
output = model(torch_inputs)

In [25]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [26]:
def training_step(inputs, targets):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    optimizer.step()
    model.zero_grad()
    return loss

##### 컴파일을 사용해 파이토치 모듈 속도 높이기

In [27]:
compiled_model = torch.compile(model)

In [28]:
@torch.compile
def dense(inputs, W, b):
    return torch.nn.relu(torch.matmul(inputs, W) + b)

###### [추가] 컴파일 효과 측정하기

책에는 없는 내용입니다. `torch.compile`은 모델을 감싸기만 하고, 실제 컴파일은 처음 호출할 때 일어납니다. 아래 셀은 일반 모델과 컴파일한 모델의 호출 시간을 직접 비교합니다.

In [29]:
# [추가] 일반 모델과 컴파일한 모델의 호출 시간을 비교합니다.
import time

def timed(fn, x, n):
    """fn(x)를 n번 호출했을 때 한 번당 걸린 시간(ms)"""
    with torch.no_grad():
        t = time.perf_counter()
        for _ in range(n):
            fn(x)
        if x.is_cuda:
            torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1e3

def compare(name, module, x, n=50):
    timed(module, x, 3)  # 준비 운동
    plain = timed(module, x, n)

    compiled = torch.compile(module)
    try:
        first = timed(compiled, x, 1)  # 실제 컴파일은 첫 호출에서 일어납니다
    except Exception as e:
        print(f"[{name}] 컴파일 실패: {type(e).__name__}: {str(e).strip().splitlines()[0][:200]}")
        return
    fast = timed(compiled, x, n)

    print(
        f"[{name}] 일반 {plain:.3f} ms | 컴파일 {fast:.3f} ms | "
        f"{plain / fast:.2f}배 | 첫 호출(컴파일 포함) {first / 1e3:.1f} s"
    )

compare("선형 모델", model, torch_inputs)

[선형 모델] 일반 0.015 ms | 컴파일 0.063 ms | 0.24배 | 첫 호출(컴파일 포함) 2.2 s


In [32]:
# [추가] 연산이 더 많은 작은 CNN으로 같은 비교를 해 봅니다.
class SmallConvNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, 3, padding=1), torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, 3, padding=1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, 3, padding=1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = torch.nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# GPU에서 재려면 "cuda"로 바꿉니다. GPU 메모리가 작으면 "cpu"로 둡니다.
device = "cpu"

convnet = SmallConvNet().to(device).eval()
images = torch.rand(32, 3, 64, 64, device=device)
compare("CNN", convnet, images, n=20)

[CNN] 일반 889.648 ms | 컴파일 409.551 ms | 2.17배 | 첫 호출(컴파일 포함) 0.8 s


#### 파이토치 방식의 차이점